## Complete Gaussian calibration table

This cell completes the precomputed Gaussian null-calibration table for the
combined $D^{\mathrm{diff}}$ statistic using separations $s=1,2$.

For each $(\alpha,N_{\mathrm{joint}})$ cell, the null distribution is described by

$$
D_{\mathrm{null}}^{\mathrm{diff}}
\sim
\mathcal{N}\!\left(
\mu(\alpha,N_{\mathrm{joint}}),
\sigma^2(\alpha,N_{\mathrm{joint}})
\right).
$$

The cell first keeps all previously valid calibration cells unchanged. For
missing cells where suitable supplemental geometries were found, it evaluates
$D^{\mathrm{diff}}$ on 500 null maps and directly measures the corresponding
$\mu$ and $\sigma$. 

If no suitable geometry exists for a missing cell, $\mu$ and the variance are
estimated from the nearest valid $N_{\mathrm{joint}}$ cells at the same
$\alpha$, using interpolation or boundary extrapolation.

The completed cells are labelled as

$$
\texttt{fit\_method}=0
\quad\text{: original direct calibration},
$$

$$
\texttt{fit\_method}=1
\quad\text{: supplemental direct calibration},
$$

$$
\texttt{fit\_method}=2
\quad\text{: interpolation/extrapolation fallback}.
$$

Finally, the cell verifies that the requested $(\alpha,N_{\mathrm{joint}})$
range is complete and saves the finished lookup table in both machine-readable
`.npz` and human-readable `.dat` formats.

## Gaussian null-calibration lookup table

Each row of the completed table corresponds to one
$(\alpha,N_{\mathrm{joint}})$ cell and stores the Gaussian null model

$$
D_{\mathrm{null}}^{\mathrm{diff}}
\mid
(\alpha,N_{\mathrm{joint}})
\sim
\mathcal{N}(\mu,\sigma^2).
$$

The columns are:

| Column | Meaning |
|---|---|
| `alpha_index` | Index of $\alpha$ in the stored $\alpha$ grid |
| `alpha_deg` | Circle opening angle $\alpha$ in degrees |
| `N_joint` | Number of jointly valid samples |
| `mu` | Mean of the null $D^{\mathrm{diff}}$ distribution |
| `sigma` | Standard deviation of the null $D^{\mathrm{diff}}$ distribution |
| `n_geometry` | Number of geometries contributing directly to this cell |
| `n_pooled_values` | Number of null $D^{\mathrm{diff}}$ values used to determine $\mu$ and $\sigma$ |
| `fit_valid` | `1` if the calibration cell is valid |
| `fit_method` | How the calibration for this cell was obtained |

The `fit_method` flag is

$$
\begin{array}{ccl}
0 &:& \text{original direct calibration},\\
1 &:& \text{supplemental direct calibration},\\
2 &:& \text{interpolated/extrapolated fallback}.
\end{array}
$$

For `fit_method = 2`, no direct geometry exists for that cell, so

$$
n_{\mathrm{geometry}}
=
n_{\mathrm{pooled\ values}}
=
0.
$$

To use the table, locate the row corresponding to the required
$(\alpha,N_{\mathrm{joint}})$ and read its $\mu$ and $\sigma$. These define
the null calibration used to standardise an observed value,

$$
Z
=
\frac{
D^{\mathrm{diff}}-\mu(\alpha,N_{\mathrm{joint}})
}{
\sigma(\alpha,N_{\mathrm{joint}})
}.
$$

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib
import healpy as hp
import numpy as np
# ==================================================
# PATHS
# ==================================================
root = Path(
    "/hydrarepo/xluo/circles-in-the-sky"
)
src = root / "src"
calibration_dir = (
    root
    / "Ddiff_gaussian_alpha_njoint_precalibration"
)
mask_file = (
    src
    / "COM_Mask_CMB-common-Mask-Int_512_R3.00.fits"
)
filter_file = (
    root
    / "SW_Doppler_ISW_filter_lmax800.dat"
)
null_map_dir = (
    root
    / "null_standardisation_500_nside512_lmax800"
)

# Original successful calibration.  Valid cells from this file
# are preserved exactly.
source_calibration_npz = (
    calibration_dir
    / "Ddiff_gaussian_fit_s12_alpha25_45_sep179p0.npz"
)

# Latest combined geometry catalogue.  The variable-separation
# search added zero new geometries, but this file contains the full
# pass-1 + pass-2 catalogue and the final found_count table.
source_geometry_npz = (
    calibration_dir
    / "Ddiff_missing_geometry_targets20_variable_sep_safe_nside512.npz"
)

# General completed lookup-table names for later blind-search use.
output_npz = (
    calibration_dir
    / "Ddiff_gaussian_fit_s12_alpha25_45_complete_nside512_lmax800_n500.npz"
)
output_table = (
    calibration_dir
    / "Ddiff_gaussian_fit_s12_alpha25_45_complete_nside512_lmax800_n500.dat"
)

# Checkpoint so an interrupted 500-null run can resume.
checkpoint_npz = (
    calibration_dir
    / "Ddiff_gaussian_supplemental_fit_checkpoint_nside512_lmax800_n500.npz"
)
CHECKPOINT_EVERY = 100


# ==================================================
# INPUT CHECKS
# ==================================================
for required_file in [
    mask_file,
    filter_file,
    source_calibration_npz,
    source_geometry_npz,
]:
    if not required_file.exists():
        raise FileNotFoundError(required_file)
null_files = sorted(
    null_map_dir.glob("null_r????.fits")
)

if len(null_files) != 500:
    raise RuntimeError(
        f"Expected 500 null maps, found {len(null_files)}."
    )

# ==================================================
# LOAD ORIGINAL GAUSSIAN CALIBRATION
# ==================================================
old = np.load(source_calibration_npz)
alpha_grid = np.asarray(
    old["alpha_grid"],
    dtype=np.float64,
)
mu_old = np.asarray(
    old["mu"],
    dtype=np.float64,
)
sigma_old = np.asarray(
    old["sigma"],
    dtype=np.float64,
)
fit_valid_old = np.asarray(
    old["fit_valid"],
    dtype=bool,
)
n_geometry_old = np.asarray(
    old["n_geometry"],
    dtype=np.int64,
)
n_pooled_old = np.asarray(
    old["n_pooled_values"],
    dtype=np.int64,
)

nside = int(old["nside"])
nside_search = int(old["nside_search"])
nphi = int(old["nphi"])
m_min = int(old["m_min"])
mask_threshold = float(old["mask_threshold"])
d_min_joint_samples = int(old["d_min_joint_samples"])
d_filter_lmax = int(old["d_filter_lmax"])
d_diff_max_separation = int(old["d_diff_max_separation"])
orientable = bool(old["orientable"])
NJOINT_MIN = int(old["njoint_min"])
NJOINT_MAX = int(old["njoint_max"])
n_alpha = len(alpha_grid)
expected_shape = (
    n_alpha,
    nphi + 1,
)
for name, array in [
    ("mu", mu_old),
    ("sigma", sigma_old),
    ("fit_valid", fit_valid_old),
    ("n_geometry", n_geometry_old),
    ("n_pooled_values", n_pooled_old),
]:
    if array.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected {name} shape: {array.shape}; "
            f"expected {expected_shape}."
        )
if not orientable:
    raise RuntimeError(
        "Expected orientable=True."
    )
if d_diff_max_separation != 2:
    raise RuntimeError(
        "Expected d_diff_max_separation=2."
    )
if d_min_joint_samples != 100:
    raise RuntimeError(
        f"Expected d_min_joint_samples=100, found "
        f"{d_min_joint_samples}."
    )
if nphi != 512:
    raise RuntimeError(
        f"Expected nphi=512, found {nphi}."
    )


# ==================================================
# DEFINE ORIGINAL MISSING CELLS
# ==================================================
requested_range = np.zeros(
    nphi + 1,
    dtype=bool,
)
requested_range[
    NJOINT_MIN:NJOINT_MAX + 1
] = True
original_target_mask = (
    (~fit_valid_old)
    & requested_range[None, :]
)
n_original_missing = int(
    np.count_nonzero(original_target_mask)
)


# ==================================================
# LOAD SUPPLEMENTAL GEOMETRY CATALOGUE
# ==================================================
geom = np.load(source_geometry_npz)
required_geometry_fields = [
    "alpha_index",
    "alpha_deg",
    "N_joint",
    "pixel_i",
    "pixel_j",
    "beta_bin",
    "beta_deg",
    "separation_deg",
    "original_target_mask",
    "found_count",
]
for field in required_geometry_fields:
    if field not in geom.files:
        raise RuntimeError(
            f"Geometry NPZ is missing field: {field}"
        )
geometry_alpha_index = np.asarray(
    geom["alpha_index"],
    dtype=np.int32,
)
geometry_alpha_deg = np.asarray(
    geom["alpha_deg"],
    dtype=np.float64,
)
geometry_n_joint = np.asarray(
    geom["N_joint"],
    dtype=np.int32,
)
geometry_pixel_i = np.asarray(
    geom["pixel_i"],
    dtype=np.int32,
)
geometry_pixel_j = np.asarray(
    geom["pixel_j"],
    dtype=np.int32,
)
geometry_beta_bin = np.asarray(
    geom["beta_bin"],
    dtype=np.int32,
)
geometry_beta_deg = np.asarray(
    geom["beta_deg"],
    dtype=np.float64,
)
geometry_separation_deg = np.asarray(
    geom["separation_deg"],
    dtype=np.float64,
)
geometry_original_target_mask = np.asarray(
    geom["original_target_mask"],
    dtype=bool,
)
found_count = np.asarray(
    geom["found_count"],
    dtype=np.int64,
)
n_geometry_records = len(geometry_alpha_index)
for name, array in [
    ("alpha_deg", geometry_alpha_deg),
    ("N_joint", geometry_n_joint),
    ("pixel_i", geometry_pixel_i),
    ("pixel_j", geometry_pixel_j),
    ("beta_bin", geometry_beta_bin),
    ("beta_deg", geometry_beta_deg),
    ("separation_deg", geometry_separation_deg),
]:
    if len(array) != n_geometry_records:
        raise RuntimeError(
            f"Geometry field {name} has inconsistent length."
        )
if geometry_original_target_mask.shape != expected_shape:
    raise RuntimeError(
        "Geometry original_target_mask has wrong shape."
    )
if found_count.shape != expected_shape:
    raise RuntimeError(
        "Geometry found_count has wrong shape."
    )
if not np.array_equal(
    geometry_original_target_mask,
    original_target_mask,
):
    raise RuntimeError(
        "Geometry catalogue target mask does not match the "
        "missing cells in the source Gaussian calibration."
    )

# Recount catalogue entries and require exact agreement with found_count.
recount = np.zeros(
    expected_shape,
    dtype=np.int64,
)
np.add.at(
    recount,
    (
        geometry_alpha_index,
        geometry_n_joint,
    ),
    1,
)
if not np.array_equal(
    recount,
    found_count,
):
    bad = np.argwhere(
        recount != found_count
    )[0]
    raise RuntimeError(
        "Geometry catalogue recount does not match found_count. "
        f"First mismatch: alpha_index={bad[0]}, "
        f"N_joint={bad[1]}, recount={recount[tuple(bad)]}, "
        f"found_count={found_count[tuple(bad)]}."
    )

# Every saved geometry must belong to an originally missing cell.
if n_geometry_records > 0:
    if not np.all(
        original_target_mask[
            geometry_alpha_index,
            geometry_n_joint,
        ]
    ):
        raise RuntimeError(
            "Geometry catalogue contains a record outside the "
            "originally missing cells."
        )
    
# Saved alpha / beta metadata must agree exactly with the indices.
if not np.allclose(
    geometry_alpha_deg,
    alpha_grid[geometry_alpha_index],
    rtol=0.0,
    atol=1.0e-10,
):
    raise RuntimeError(
        "Saved geometry alpha_deg does not match alpha_grid."
    )
expected_beta_deg = (
    360.0
    * geometry_beta_bin.astype(np.float64)
    / nphi
)
if not np.allclose(
    geometry_beta_deg,
    expected_beta_deg,
    rtol=0.0,
    atol=1.0e-10,
):
    raise RuntimeError(
        "Saved geometry beta_deg does not match beta_bin."
    )
if np.any(
    (geometry_beta_bin < 0)
    | (geometry_beta_bin >= nphi)
):
    raise RuntimeError(
        "Geometry catalogue contains an invalid beta_bin."
    )

# ==================================================
# IDENTIFY DIRECT-SUPPLEMENTAL VS FALLBACK CELLS
# ==================================================
supplemental_direct_mask = (
    original_target_mask
    & (found_count > 0)
)
zero_geometry_mask = (
    original_target_mask
    & (found_count == 0)
)
# Zero-geometry cells are handled by the generic fallback below.
zero_locations = np.argwhere(
    zero_geometry_mask
)
n_supplemental_cells = int(
    np.count_nonzero(
        supplemental_direct_mask
    )
)
n_zero_cells = int(
    np.count_nonzero(
        zero_geometry_mask
    )
)

print("======================================")
print("SUPPLEMENTAL GAUSSIAN COMPLETION")
print("======================================")
print(f"alpha values                 : {n_alpha}")
print(f"N_joint range                : {NJOINT_MIN} to {NJOINT_MAX}")
print(f"original missing cells       : {n_original_missing}")
print(f"direct supplemental cells    : {n_supplemental_cells}")
print(f"fallback cells               : {n_zero_cells}")
print(f"saved supplemental geometries: {n_geometry_records}")
print(f"null maps                    : {len(null_files)}")
print()


# ==================================================
# READ MASK AND D FILTER
# ==================================================
obs_mask = hp.read_map(
    mask_file,
    field=0,
    dtype=np.float64,
)
if hp.get_nside(obs_mask) != nside:
    raise RuntimeError(
        f"Mask NSIDE={hp.get_nside(obs_mask)}, expected {nside}."
    )
filter_table = np.loadtxt(
    filter_file,
    comments="#",
)
d_filter = np.zeros(
    d_filter_lmax + 1,
    dtype=np.float64,
)
filter_ell = filter_table[:, 0].astype(int)
filter_value = np.asarray(
    filter_table[:, 1],
    dtype=np.float64,
)
inside = (
    (filter_ell >= 0)
    & (filter_ell <= d_filter_lmax)
)
d_filter[
    filter_ell[inside]
] = filter_value[inside]
def apply_d_filter(input_map):
    """Exactly the same map filter used in the original calibration."""
    alm = hp.map2alm(
        input_map,
        lmax=d_filter_lmax,
    )
    filtered_alm = hp.almxfl(
        alm,
        d_filter,
    )
    return hp.alm2map(
        filtered_alm,
        nside=nside,
        lmax=d_filter_lmax,
    )


# ==================================================
# CIRCLE GEOMETRY
# ==================================================
phi_ring = (
    2.0
    * np.pi
    * np.arange(
        1,
        nphi + 1,
        dtype=np.float64,
    )
    / nphi
)
def circle_ring_pixels(
    pixel,
    alpha_deg,
):
    theta, phi = hp.pix2ang(
        nside_search,
        pixel,
        nest=False,
    )
    alpha = np.deg2rad(
        alpha_deg
    )
    rotation_matrix = np.array(
        [
            [
                np.cos(theta) * np.cos(phi),
                -np.sin(phi),
                np.sin(theta) * np.cos(phi),
            ],
            [
                np.cos(theta) * np.sin(phi),
                np.cos(phi),
                np.sin(theta) * np.sin(phi),
            ],
            [
                -np.sin(theta),
                0.0,
                np.cos(theta),
            ],
        ],
        dtype=np.float64,
    )
    local_vectors = np.vstack(
        [
            np.sin(alpha) * np.cos(phi_ring),
            np.sin(alpha) * np.sin(phi_ring),
            np.full(
                nphi,
                np.cos(alpha),
                dtype=np.float64,
            ),
        ]
    )
    sky_vectors = (
        rotation_matrix
        @ local_vectors
    )
    theta_ring = np.arccos(
        np.clip(
            sky_vectors[2],
            -1.0,
            1.0,
        )
    )
    phi_ring_sky = (
        np.arctan2(
            sky_vectors[1],
            sky_vectors[0],
        )
        + np.pi
    )
    return hp.ang2pix(
        nside,
        theta_ring,
        phi_ring_sky,
        nest=False,
    )


# ==================================================
# PRECOMPUTE THE RING PIXELS / MASK VALIDITY USED BY
# THE SUPPLEMENTAL GEOMETRY CATALOGUE
# ==================================================
circle_keys = set()
for alpha_index, pixel_i, pixel_j in zip(
    geometry_alpha_index,
    geometry_pixel_i,
    geometry_pixel_j,
):
    circle_keys.add(
        (
            int(alpha_index),
            int(pixel_i),
        )
    )
    circle_keys.add(
        (
            int(alpha_index),
            int(pixel_j),
        )
    )
circle_cache = {}
for cache_index, (
    alpha_index,
    pixel,
) in enumerate(
    sorted(circle_keys)
):
    ring_pixels = circle_ring_pixels(
        pixel,
        float(alpha_grid[alpha_index]),
    )
    valid = (
        obs_mask[ring_pixels]
        >= mask_threshold
    )
    circle_cache[
        (alpha_index, pixel)
    ] = (
        ring_pixels,
        valid,
    )
    if (
        cache_index == 0
        or (cache_index + 1) % 1000 == 0
        or cache_index + 1 == len(circle_keys)
    ):
        print(
            f"Precomputed circles: {cache_index + 1}/"
            f"{len(circle_keys)}"
        )
print()


# ==================================================
# VERIFY EVERY SAVED (i,j,beta|alpha) REALLY HAS THE
# SAVED N_joint, USING A DIRECT MASK-ONLY CALCULATION.
# ==================================================
index_ring = np.arange(
    nphi,
    dtype=np.int64,
)
def n_joint_at_beta_from_validity(
    valid_i,
    valid_j,
    beta_bin,
):
    partner = (
        beta_bin
        - index_ring
    ) % nphi
    n_joint = nphi
    for separation_bin in range(
        1,
        d_diff_max_separation + 1,
    ):
        valid_delta_i = (
            valid_i
            & np.roll(
                valid_i,
                -separation_bin,
            )
        )
        valid_delta_j = (
            valid_j
            & np.roll(
                valid_j,
                separation_bin,
            )
        )
        n_joint_s = int(
            np.count_nonzero(
                valid_delta_i
                & valid_delta_j[partner]
            )
        )
        n_joint = min(
            n_joint,
            n_joint_s,
        )
    return n_joint
for record_index in range(
    n_geometry_records
):
    alpha_index = int(
        geometry_alpha_index[record_index]
    )
    pixel_i = int(
        geometry_pixel_i[record_index]
    )
    pixel_j = int(
        geometry_pixel_j[record_index]
    )
    beta_bin = int(
        geometry_beta_bin[record_index]
    )
    target_n = int(
        geometry_n_joint[record_index]
    )
    _, valid_i = circle_cache[
        (alpha_index, pixel_i)
    ]
    _, valid_j = circle_cache[
        (alpha_index, pixel_j)
    ]
    actual_n = n_joint_at_beta_from_validity(
        valid_i,
        valid_j,
        beta_bin,
    )
    if actual_n != target_n:
        raise RuntimeError(
            "Saved supplemental geometry failed direct N_joint "
            "verification: "
            f"record={record_index}, alpha_index={alpha_index}, "
            f"N_saved={target_n}, N_actual={actual_n}, "
            f"pixel_i={pixel_i}, pixel_j={pixel_j}, "
            f"beta_bin={beta_bin}."
        )
print(
    "Supplemental geometry N_joint verification: PASS "
    f"({n_geometry_records} records)"
)
print()


# ==================================================
# GROUP SAVED RECORDS BY (alpha_index, pixel_i, pixel_j)
# SO ONE CIRCLE-PAIR EVALUATION CAN SERVE EVERY SAVED
# BETA FOR THAT PAIR / ALPHA.
# ==================================================
group_records = defaultdict(list)
for record_index in range(
    n_geometry_records
):
    key = (
        int(geometry_alpha_index[record_index]),
        int(geometry_pixel_i[record_index]),
        int(geometry_pixel_j[record_index]),
    )
    group_records[key].append(
        record_index
    )
# Deterministic processing order.
groups = []
for key in sorted(group_records):
    record_indices = np.asarray(
        group_records[key],
        dtype=np.int64,
    )
    # Within a given pair+alpha, each target cell should occur once.
    n_values = geometry_n_joint[
        record_indices
    ]
    if len(np.unique(n_values)) != len(n_values):
        raise RuntimeError(
            "Duplicate N_joint target within one "
            "(alpha,pixel_i,pixel_j) group."
        )
    groups.append(
        (
            key,
            record_indices,
        )
    )
print(
    f"Unique supplemental (alpha,i,j) groups: {len(groups)}"
)
print()


# ==================================================
# TEMPERATURE RING PROCESSING
# ==================================================
def processed_ring_from_cache(
    sky_map,
    alpha_index,
    pixel,
):
    ring_pixels, valid = circle_cache[
        (alpha_index, pixel)
    ]
    temperature = np.asarray(
        sky_map[ring_pixels],
        dtype=np.float64,
    )
    # Same low-mode removal as the original calibration.
    modes = np.fft.rfft(
        temperature
    )
    modes[:m_min] = 0.0
    temperature = np.fft.irfft(
        modes,
        n=nphi,
    )
    return temperature, valid


# ==================================================
# DIRECT D_diff AT ONLY THE SAVED BETA BINS
#
# This is algebraically identical to selecting those beta bins
# from the full FFT D_diff curve, but avoids calculating all 512
# beta values when only a few are needed.
# ==================================================
def get_d_diff_at_selected_betas(
    ring_i,
    valid_i,
    ring_j,
    valid_j,
    beta_bins,
):
    beta_bins = np.asarray(
        beta_bins,
        dtype=np.int64,
    )
    n_beta = len(beta_bins)
    numerator_total = np.zeros(
        n_beta,
        dtype=np.float64,
    )
    denominator_total = np.zeros(
        n_beta,
        dtype=np.float64,
    )
    n_joint = np.full(
        n_beta,
        nphi,
        dtype=np.int16,
    )
    partners = (
        beta_bins[:, None]
        - index_ring[None, :]
    ) % nphi
    for separation_bin in range(
        1,
        d_diff_max_separation + 1,
    ):
        delta_t_i = (
            np.roll(
                ring_i,
                -separation_bin,
            )
            - ring_i
        )
        valid_delta_i = (
            valid_i
            & np.roll(
                valid_i,
                -separation_bin,
            )
        )
        # Current calibration is orientable=True.
        delta_t_j = (
            np.roll(
                ring_j,
                separation_bin,
            )
            - ring_j
        )
        valid_delta_j = (
            valid_j
            & np.roll(
                valid_j,
                separation_bin,
            )
        )
        for beta_local in range(
            n_beta
        ):
            partner = partners[
                beta_local
            ]
            joint = (
                valid_delta_i
                & valid_delta_j[partner]
            )
            n_joint_s = int(
                np.count_nonzero(joint)
            )
            n_joint[beta_local] = min(
                int(n_joint[beta_local]),
                n_joint_s,
            )
            if n_joint_s == 0:
                continue
            partner_joint = partner[
                joint
            ]
            delta_i_joint = delta_t_i[
                joint
            ]
            delta_j_joint = delta_t_j[
                partner_joint
            ]
            numerator_total[
                beta_local
            ] += (
                2.0
                * np.sum(
                    delta_i_joint
                    * delta_j_joint
                )
            )
            denominator_total[
                beta_local
            ] += np.sum(
                delta_i_joint**2
                + delta_j_joint**2
            )
    d_diff = np.full(
        n_beta,
        np.nan,
        dtype=np.float64,
    )
    usable = (
        (n_joint >= d_min_joint_samples)
        & (
            denominator_total
            > np.finfo(np.float64).tiny
        )
    )
    d_diff[usable] = np.clip(
        numerator_total[usable]
        / denominator_total[usable],
        -1.0,
        1.0,
    )
    return d_diff, n_joint


# ==================================================
# FULL FFT REFERENCE CURVE, USED ONLY FOR A STARTUP
# SELF-CHECK OF THE DIRECT SELECTED-BETA EVALUATOR.
# ==================================================
def get_d_diff_curve_fft_from_rings(
    ring_i,
    valid_i,
    ring_j,
    valid_j,
):
    numerator_total = np.zeros(
        nphi,
        dtype=np.float64,
    )
    denominator_total = np.zeros(
        nphi,
        dtype=np.float64,
    )
    n_joint = np.full(
        nphi,
        nphi,
        dtype=np.int16,
    )
    for separation_bin in range(
        1,
        d_diff_max_separation + 1,
    ):
        delta_t_i = (
            np.roll(
                ring_i,
                -separation_bin,
            )
            - ring_i
        )
        valid_delta_i = (
            valid_i
            & np.roll(
                valid_i,
                -separation_bin,
            )
        )
        delta_t_j = (
            np.roll(
                ring_j,
                separation_bin,
            )
            - ring_j
        )
        valid_delta_j = (
            valid_j
            & np.roll(
                valid_j,
                separation_bin,
            )
        )
        w_i = valid_delta_i.astype(
            np.float64
        )
        w_j = valid_delta_j.astype(
            np.float64
        )
        fw_i = np.fft.fft(w_i)
        fw_j = np.fft.fft(w_j)
        fx_i = np.fft.fft(
            w_i * delta_t_i
        )
        fx_j = np.fft.fft(
            w_j * delta_t_j
        )
        fe_i = np.fft.fft(
            w_i * delta_t_i**2
        )
        fe_j = np.fft.fft(
            w_j * delta_t_j**2
        )
        numerator_total += (
            2.0
            * np.fft.ifft(
                fx_i * fx_j
            ).real
        )
        denominator_total += np.fft.ifft(
            fe_i * fw_j
            + fw_i * fe_j
        ).real
        n_joint_s = np.rint(
            np.fft.ifft(
                fw_i * fw_j
            ).real
        ).astype(np.int16)
        n_joint_s = np.clip(
            n_joint_s,
            0,
            nphi,
        )
        n_joint = np.minimum(
            n_joint,
            n_joint_s,
        )
    d_diff = np.full(
        nphi,
        np.nan,
        dtype=np.float64,
    )
    usable = (
        (n_joint >= d_min_joint_samples)
        & (
            denominator_total
            > np.finfo(np.float64).tiny
        )
    )
    d_diff[usable] = np.clip(
        numerator_total[usable]
        / denominator_total[usable],
        -1.0,
        1.0,
    )
    return d_diff, n_joint


# ==================================================
# GEOMETRY SIGNATURE FOR SAFE CHECKPOINT RESUME
# ==================================================
signature_hasher = hashlib.sha256()
for array in [
    geometry_alpha_index,
    geometry_n_joint,
    geometry_pixel_i,
    geometry_pixel_j,
    geometry_beta_bin,
]:
    signature_hasher.update(
        np.ascontiguousarray(array).tobytes()
    )
geometry_signature = (
    signature_hasher.hexdigest()
)


# ==================================================
# SUPPLEMENTAL ONLINE MOMENTS
# ==================================================
supp_count = np.zeros(
    expected_shape,
    dtype=np.int64,
)
supp_sum = np.zeros(
    expected_shape,
    dtype=np.float64,
)
supp_sumsq = np.zeros(
    expected_shape,
    dtype=np.float64,
)
start_null_index = 0
if checkpoint_npz.exists():
    checkpoint = np.load(
        checkpoint_npz
    )
    checkpoint_signature = str(
        checkpoint["geometry_signature"]
    )
    if checkpoint_signature != geometry_signature:
        raise RuntimeError(
            "Checkpoint geometry signature does not match the "
            "current supplemental geometry catalogue."
        )
    supp_count = np.asarray(
        checkpoint["count"],
        dtype=np.int64,
    )
    supp_sum = np.asarray(
        checkpoint["sum"],
        dtype=np.float64,
    )
    supp_sumsq = np.asarray(
        checkpoint["sumsq"],
        dtype=np.float64,
    )
    start_null_index = int(
        checkpoint["next_null_index"]
    )
    if supp_count.shape != expected_shape:
        raise RuntimeError(
            "Checkpoint count shape is wrong."
        )
    if supp_sum.shape != expected_shape:
        raise RuntimeError(
            "Checkpoint sum shape is wrong."
        )
    if supp_sumsq.shape != expected_shape:
        raise RuntimeError(
            "Checkpoint sumsq shape is wrong."
        )
    if not (
        0 <= start_null_index <= len(null_files)
    ):
        raise RuntimeError(
            "Checkpoint next_null_index is invalid."
        )
    print(
        f"Resuming checkpoint at null map "
        f"{start_null_index + 1}/{len(null_files)}"
        if start_null_index < len(null_files)
        else "Checkpoint already contains all null maps."
    )
    print()


# ==================================================
# FIRST-NULL SELF-CHECK:
# DIRECT SELECTED-BETA D_diff == FULL FFT D_diff
# ==================================================
if start_null_index == 0:
    first_null_map = hp.read_map(
        null_files[0],
        field=0,
        dtype=np.float64,
    )
    first_filtered_null = apply_d_filter(
        first_null_map
    )
    n_check_groups = min(
        10,
        len(groups),
    )
    for check_group_index in range(
        n_check_groups
    ):
        (
            alpha_index,
            pixel_i,
            pixel_j,
        ), record_indices = groups[
            check_group_index
        ]
        ring_i, valid_i = processed_ring_from_cache(
            first_filtered_null,
            alpha_index,
            pixel_i,
        )
        ring_j, valid_j = processed_ring_from_cache(
            first_filtered_null,
            alpha_index,
            pixel_j,
        )
        beta_bins = geometry_beta_bin[
            record_indices
        ]
        direct_d, direct_n = get_d_diff_at_selected_betas(
            ring_i,
            valid_i,
            ring_j,
            valid_j,
            beta_bins,
        )
        fft_d, fft_n = get_d_diff_curve_fft_from_rings(
            ring_i,
            valid_i,
            ring_j,
            valid_j,
        )
        target_n = geometry_n_joint[
            record_indices
        ]
        if not np.array_equal(
            direct_n,
            target_n,
        ):
            raise RuntimeError(
                "Direct selected-beta N_joint self-check failed."
            )
        if not np.array_equal(
            fft_n[beta_bins],
            target_n,
        ):
            raise RuntimeError(
                "FFT-reference N_joint self-check failed."
            )
        if not np.allclose(
            direct_d,
            fft_d[beta_bins],
            rtol=1.0e-12,
            atol=1.0e-12,
            equal_nan=True,
        ):
            raise RuntimeError(
                "Direct selected-beta D_diff does not match the "
                "full FFT reference curve."
            )
    print(
        f"Direct-vs-FFT D_diff self-check: PASS "
        f"({n_check_groups} groups)"
    )
    print()
else:
    first_filtered_null = None


# ==================================================
# 500 NULL MAPS: FIT ONLY THE SUPPLEMENTAL DIRECT CELLS
# ==================================================
for null_index in range(
    start_null_index,
    len(null_files),
):
    if (
        null_index == 0
        and first_filtered_null is not None
    ):
        filtered_null_map = (
            first_filtered_null
        )
    else:
        null_map = hp.read_map(
            null_files[null_index],
            field=0,
            dtype=np.float64,
        )
        filtered_null_map = apply_d_filter(
            null_map
        )
    for (
        alpha_index,
        pixel_i,
        pixel_j,
    ), record_indices in groups:
        ring_i, valid_i = processed_ring_from_cache(
            filtered_null_map,
            alpha_index,
            pixel_i,
        )
        ring_j, valid_j = processed_ring_from_cache(
            filtered_null_map,
            alpha_index,
            pixel_j,
        )
        beta_bins = geometry_beta_bin[
            record_indices
        ]
        target_n = geometry_n_joint[
            record_indices
        ]
        d_values, actual_n = get_d_diff_at_selected_betas(
            ring_i,
            valid_i,
            ring_j,
            valid_j,
            beta_bins,
        )
        # Mask geometry is map-independent, so this must always agree.
        if not np.array_equal(
            actual_n,
            target_n,
        ):
            raise RuntimeError(
                "N_joint changed unexpectedly during null processing."
            )
        finite = np.isfinite(
            d_values
        )
        if not np.any(finite):
            continue
        n_use = target_n[
            finite
        ].astype(
            np.int64,
            copy=False,
        )
        d_use = d_values[
            finite
        ]
        # A group contains at most one record for each N_joint,
        # so direct indexed addition is safe here.
        supp_count[
            alpha_index,
            n_use,
        ] += 1
        supp_sum[
            alpha_index,
            n_use,
        ] += d_use
        supp_sumsq[
            alpha_index,
            n_use,
        ] += d_use**2
    completed_nulls = (
        null_index + 1
    )
    if (
        completed_nulls == 1
        or completed_nulls % CHECKPOINT_EVERY == 0
        or completed_nulls == len(null_files)
    ):
        np.savez_compressed(
            checkpoint_npz,
            geometry_signature=np.array(
                geometry_signature
            ),
            next_null_index=np.array(
                completed_nulls
            ),
            count=supp_count,
            sum=supp_sum,
            sumsq=supp_sumsq,
            source_geometry_npz=np.array(
                str(source_geometry_npz)
            ),
            source_calibration_npz=np.array(
                str(source_calibration_npz)
            ),
        )
        print(
            f"Null maps: {completed_nulls}/{len(null_files)} "
            f"(checkpoint saved)"
        )


# ==================================================
# FIT SUPPLEMENTAL DIRECT GAUSSIANS
# ==================================================
supp_mu = np.full(
    expected_shape,
    np.nan,
    dtype=np.float64,
)
supp_sigma = np.full(
    expected_shape,
    np.nan,
    dtype=np.float64,
)
supp_valid = (
    supplemental_direct_mask
    & (supp_count >= 3)
)
supp_mu[supp_valid] = (
    supp_sum[supp_valid]
    / supp_count[supp_valid]
)
supp_variance = np.full(
    expected_shape,
    np.nan,
    dtype=np.float64,
)
supp_variance[supp_valid] = (
    supp_sumsq[supp_valid]
    / supp_count[supp_valid]
    - supp_mu[supp_valid]**2
)
# Remove only tiny negative floating-point roundoff.
supp_variance[supp_valid] = np.maximum(
    supp_variance[supp_valid],
    0.0,
)
supp_sigma[supp_valid] = np.sqrt(
    supp_variance[supp_valid]
)
supp_valid &= (
    np.isfinite(supp_mu)
    & np.isfinite(supp_sigma)
    & (supp_sigma > 0.0)
)
missing_direct_fit = (
    supplemental_direct_mask
    & (~supp_valid)
)
if np.any(missing_direct_fit):
    bad = np.argwhere(
        missing_direct_fit
    )[0]
    raise RuntimeError(
        "A reachable supplemental cell failed Gaussian fitting: "
        f"alpha_index={bad[0]}, N_joint={bad[1]}, "
        f"count={supp_count[tuple(bad)]}."
    )


# ==================================================
# BUILD COMPLETE TABLE
#
# fit_method:
#   0 = original direct fit
#   1 = supplemental direct fit
#   2 = interpolated/extrapolated fallback for zero-geometry cells
# ==================================================
mu_final = mu_old.copy()
sigma_final = sigma_old.copy()
fit_valid_final = fit_valid_old.copy()
n_geometry_final = n_geometry_old.copy()
n_pooled_final = n_pooled_old.copy()
fit_method = np.full(
    expected_shape,
    -1,
    dtype=np.int8,
)
fit_method[
    fit_valid_old
    & requested_range[None, :]
] = 0
# Insert direct supplemental fits.
mu_final[
    supplemental_direct_mask
] = supp_mu[
    supplemental_direct_mask
]
sigma_final[
    supplemental_direct_mask
] = supp_sigma[
    supplemental_direct_mask
]
fit_valid_final[
    supplemental_direct_mask
] = True
n_geometry_final[
    supplemental_direct_mask
] = found_count[
    supplemental_direct_mask
]
n_pooled_final[
    supplemental_direct_mask
] = supp_count[
    supplemental_direct_mask
]
fit_method[
    supplemental_direct_mask
] = 1


# ==================================================
# FALLBACK FOR ZERO-GEOMETRY CELLS
#
# Use the nearest valid direct/supplemental N_joint cells at the same alpha.
# If valid cells exist on both sides, interpolate linearly in N_joint.
# At a boundary (for example N_joint=512), use the two nearest valid cells
# on the available side for linear extrapolation.
#
# The Gaussian mean and variance are interpolated/extrapolated separately.
# Fallback cells are explicitly marked fit_method=2 and have zero direct
# geometry / pooled-null counts.
# ==================================================
for alpha_index, n_joint_value in zero_locations:
    alpha_index = int(alpha_index)
    n_joint_value = int(n_joint_value)
    direct_valid = (
        fit_valid_final[alpha_index]
        & (fit_method[alpha_index] >= 0)
        & (fit_method[alpha_index] <= 1)
    )
    valid_n = np.flatnonzero(
        direct_valid
        & requested_range
    )
    below = valid_n[
        valid_n < n_joint_value
    ]
    above = valid_n[
        valid_n > n_joint_value
    ]
    if len(below) > 0 and len(above) > 0:
        n0 = int(below[-1])
        n1 = int(above[0])
    elif len(below) >= 2:
        n0 = int(below[-2])
        n1 = int(below[-1])
    elif len(above) >= 2:
        n0 = int(above[0])
        n1 = int(above[1])
    else:
        raise RuntimeError(
            "Cannot construct fallback because fewer than two "
            "direct valid N_joint cells are available at "
            f"alpha_index={alpha_index}, N_joint={n_joint_value}."
        )
    weight = (
        (n_joint_value - n0)
        / float(n1 - n0)
    )
    mu0 = mu_final[
        alpha_index,
        n0,
    ]
    mu1 = mu_final[
        alpha_index,
        n1,
    ]
    var0 = sigma_final[
        alpha_index,
        n0,
    ]**2
    var1 = sigma_final[
        alpha_index,
        n1,
    ]**2
    mu_fallback = (
        (1.0 - weight) * mu0
        + weight * mu1
    )
    variance_fallback = (
        (1.0 - weight) * var0
        + weight * var1
    )
    if not (
        np.isfinite(mu_fallback)
        and np.isfinite(variance_fallback)
        and variance_fallback > 0.0
    ):
        raise RuntimeError(
            "Invalid fallback Gaussian at "
            f"alpha_index={alpha_index}, "
            f"N_joint={n_joint_value}, "
            f"sources=({n0},{n1}), "
            f"variance={variance_fallback}."
        )
    mu_final[
        alpha_index,
        n_joint_value,
    ] = mu_fallback
    sigma_final[
        alpha_index,
        n_joint_value,
    ] = np.sqrt(
        variance_fallback
    )
    fit_valid_final[
        alpha_index,
        n_joint_value,
    ] = True
    n_geometry_final[
        alpha_index,
        n_joint_value,
    ] = 0
    n_pooled_final[
        alpha_index,
        n_joint_value,
    ] = 0
    fit_method[
        alpha_index,
        n_joint_value,
    ] = 2


# ==================================================
# FINAL COMPLETENESS CHECK
# ==================================================
final_requested_valid = fit_valid_final[
    :,
    NJOINT_MIN:NJOINT_MAX + 1,
]
if not np.all(
    final_requested_valid
):
    bad_local = np.argwhere(
        ~final_requested_valid
    )[0]
    bad_alpha = int(
        bad_local[0]
    )
    bad_n = int(
        NJOINT_MIN
        + bad_local[1]
    )
    raise RuntimeError(
        "Final table is not complete: "
        f"alpha_index={bad_alpha}, N_joint={bad_n}."
    )
for array_name, array in [
    ("mu", mu_final),
    ("sigma", sigma_final),
]:
    requested_values = array[
        :,
        NJOINT_MIN:NJOINT_MAX + 1,
    ]
    if not np.all(
        np.isfinite(requested_values)
    ):
        raise RuntimeError(
            f"Final {array_name} contains non-finite requested values."
        )
if not np.all(
    sigma_final[
        :,
        NJOINT_MIN:NJOINT_MAX + 1,
    ] > 0.0
):
    raise RuntimeError(
        "Final sigma contains a non-positive requested value."
    )
if np.any(
    fit_method[
        :,
        NJOINT_MIN:NJOINT_MAX + 1,
    ] < 0
):
    raise RuntimeError(
        "Final table has an unassigned fit_method."
    )


# ==================================================
# SAVE COMPLETE MACHINE-READABLE TABLE
# ==================================================
# Preserve the original metadata, then replace the lookup arrays
# with the completed versions and add explicit provenance fields.
payload = {
    key: old[key]
    for key in old.files
}
payload.update(
    {
        "mu": mu_final,
        "sigma": sigma_final,
        "fit_valid": fit_valid_final,
        "n_geometry": n_geometry_final,
        "n_pooled_values": n_pooled_final,
        "fit_method": fit_method,
        "source_fit_valid": fit_valid_old,
        "source_mu": mu_old,
        "source_sigma": sigma_old,
        "supplemental_count": supp_count,
        "supplemental_sum": supp_sum,
        "supplemental_sumsq": supp_sumsq,
        "supplemental_geometry_count": found_count,
        "source_calibration_npz": np.array(
            str(source_calibration_npz)
        ),
        "source_geometry_npz": np.array(
            str(source_geometry_npz)
        ),
        "completion_method": np.array(
            "original direct fits + supplemental direct fits + "
            "nearest-valid linear fallback for zero-geometry cells"
        ),
        "fit_method_0": np.array(
            "original_direct"
        ),
        "fit_method_1": np.array(
            "supplemental_direct"
        ),
        "fit_method_2": np.array(
            "nearest_valid_linear_fallback"
        ),
        "fallback_interpolates": np.array(
            "mu and variance"
        ),
        "completed_n_null_maps": np.array(
            len(null_files)
        ),
        "completed_table": np.array(True),
    }
)
np.savez_compressed(
    output_npz,
    **payload,
)


# ==================================================
# SAVE HUMAN-READABLE COMPLETE TABLE
# ==================================================
rows = []
for alpha_index, alpha_deg in enumerate(
    alpha_grid
):
    for n_joint_value in range(
        NJOINT_MIN,
        NJOINT_MAX + 1,
    ):
        rows.append(
            [
                alpha_index,
                alpha_deg,
                n_joint_value,
                mu_final[
                    alpha_index,
                    n_joint_value,
                ],
                sigma_final[
                    alpha_index,
                    n_joint_value,
                ],
                n_geometry_final[
                    alpha_index,
                    n_joint_value,
                ],
                n_pooled_final[
                    alpha_index,
                    n_joint_value,
                ],
                int(
                    fit_valid_final[
                        alpha_index,
                        n_joint_value,
                    ]
                ),
                int(
                    fit_method[
                        alpha_index,
                        n_joint_value,
                    ]
                ),
            ]
        )
rows = np.asarray(
    rows,
    dtype=np.float64,
)
np.savetxt(
    output_table,
    rows,
    fmt=[
        "%d",
        "%.12f",
        "%d",
        "%.16e",
        "%.16e",
        "%d",
        "%d",
        "%d",
        "%d",
    ],
    header=(
        "alpha_index alpha_deg N_joint mu sigma "
        "n_geometry n_pooled_values fit_valid fit_method\n"
        "fit_method: 0=original_direct, 1=supplemental_direct, "
        "2=nearest_valid_linear_fallback\n"
        "For fit_method=2, n_geometry=n_pooled_values=0 by design."
    ),
)


# ==================================================
# CONCISE FINAL SUMMARY
# ==================================================
method_counts = {
    method: int(
        np.count_nonzero(
            fit_method[
                :,
                NJOINT_MIN:NJOINT_MAX + 1,
            ]
            == method
        )
    )
    for method in (0, 1, 2)
}
supp_geometry_counts = found_count[
    supplemental_direct_mask
]
supp_pooled_counts = supp_count[
    supplemental_direct_mask
]
print()
print("======================================")
print("COMPLETE GAUSSIAN LOOKUP SAVED")
print("======================================")
print(
    f"requested cells       : "
    f"{n_alpha * (NJOINT_MAX - NJOINT_MIN + 1)}"
)
print(
    f"original direct       : {method_counts[0]}"
)
print(
    f"supplemental direct   : {method_counts[1]}"
)
print(
    f"fallback zero-geometry: {method_counts[2]}"
)
print(
    "supplemental geometries/cell min/median/max: "
    f"{int(np.min(supp_geometry_counts))}/"
    f"{float(np.median(supp_geometry_counts)):.1f}/"
    f"{int(np.max(supp_geometry_counts))}"
)
print(
    "supplemental null values/cell min/median/max: "
    f"{int(np.min(supp_pooled_counts))}/"
    f"{float(np.median(supp_pooled_counts)):.1f}/"
    f"{int(np.max(supp_pooled_counts))}"
)
print("all N_joint=100..512 complete: YES")
print()
print("Saved NPZ  :", output_npz)
print("Saved table:", output_table)
print("Checkpoint :", checkpoint_npz)

## Complete Gaussian null-calibration lookup

This cell completes the Gaussian null-calibration table for the combined
$D^{\mathrm{diff}}$ statistic using separations $s=1,2$.

For each $(\alpha,N_{\mathrm{joint}})$, the null distribution is represented as

$$
D_{\mathrm{null}}^{\mathrm{diff}}
\mid
(\alpha,N_{\mathrm{joint}})
\sim
\mathcal{N}\!\left(
\mu(\alpha,N_{\mathrm{joint}}),
\sigma^2(\alpha,N_{\mathrm{joint}})
\right).
$$

Previously valid calibration cells are preserved unchanged. Missing cells with
available supplemental geometries are calibrated directly using the 500 null
maps. 

Missing cells for which no suitable geometry was found are filled at fixed
$\alpha$ from the nearest directly calibrated $N_{\mathrm{joint}}$ values,
using interpolation or boundary extrapolation.

The final table records whether each cell comes from the original direct fit,
a supplemental direct fit, or the interpolation/extrapolation fallback. The
completed lookup table is then checked for completeness and saved for use in
the later blind search.

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib
import healpy as hp
import numpy as np


# ==================================================
# PATHS
# ==================================================
root = Path(
    "/hydrarepo/xluo/circles-in-the-sky"
)
src = root / "src"
calibration_dir = (
    root
    / "Ddiff_gaussian_alpha_njoint_precalibration"
)
mask_file = (
    src
    / "COM_Mask_CMB-common-Mask-Int_512_R3.00.fits"
)
filter_file = (
    root
    / "SW_Doppler_ISW_filter_lmax800.dat"
)
null_map_dir = (
    root
    / "null_standardisation_500_nside512_lmax800"
)

# Original successful calibration.  Valid cells from this file
# are preserved exactly.
source_calibration_npz = (
    calibration_dir
    / "Ddiff_gaussian_fit_s12_alpha25_45_sep179p0.npz"
)

# Latest combined geometry catalogue.  The variable-separation
# search added zero new geometries, but this file contains the full
# pass-1 + pass-2 catalogue and the final found_count table.
source_geometry_npz = (
    calibration_dir
    / "Ddiff_missing_geometry_targets20_variable_sep_safe_nside512.npz"
)

# General completed lookup-table names for later blind-search use.
output_npz = (
    calibration_dir
    / "Ddiff_gaussian_fit_s12_alpha25_45_complete_nside512_lmax800_n500.npz"
)
output_table = (
    calibration_dir
    / "Ddiff_gaussian_fit_s12_alpha25_45_complete_nside512_lmax800_n500.dat"
)

# Checkpoint so an interrupted 500-null run can resume.
checkpoint_npz = (
    calibration_dir
    / "Ddiff_gaussian_supplemental_fit_checkpoint_nside512_lmax800_n500.npz"
)

CHECKPOINT_EVERY = 100


# ==================================================
# INPUT CHECKS
# ==================================================
for required_file in [
    mask_file,
    filter_file,
    source_calibration_npz,
    source_geometry_npz,
]:
    if not required_file.exists():
        raise FileNotFoundError(required_file)
null_files = sorted(
    null_map_dir.glob("null_r????.fits")
)
if len(null_files) != 500:
    raise RuntimeError(
        f"Expected 500 null maps, found {len(null_files)}."
    )


# ==================================================
# LOAD ORIGINAL GAUSSIAN CALIBRATION
# ==================================================
old = np.load(source_calibration_npz)

alpha_grid = np.asarray(
    old["alpha_grid"],
    dtype=np.float64,
)
mu_old = np.asarray(
    old["mu"],
    dtype=np.float64,
)
sigma_old = np.asarray(
    old["sigma"],
    dtype=np.float64,
)
fit_valid_old = np.asarray(
    old["fit_valid"],
    dtype=bool,
)
n_geometry_old = np.asarray(
    old["n_geometry"],
    dtype=np.int64,
)
n_pooled_old = np.asarray(
    old["n_pooled_values"],
    dtype=np.int64,
)

nside = int(old["nside"])
nside_search = int(old["nside_search"])
nphi = int(old["nphi"])
m_min = int(old["m_min"])
mask_threshold = float(old["mask_threshold"])
d_min_joint_samples = int(old["d_min_joint_samples"])
d_filter_lmax = int(old["d_filter_lmax"])
d_diff_max_separation = int(old["d_diff_max_separation"])
orientable = bool(old["orientable"])
NJOINT_MIN = int(old["njoint_min"])
NJOINT_MAX = int(old["njoint_max"])
n_alpha = len(alpha_grid)
expected_shape = (
    n_alpha,
    nphi + 1,
)

for name, array in [
    ("mu", mu_old),
    ("sigma", sigma_old),
    ("fit_valid", fit_valid_old),
    ("n_geometry", n_geometry_old),
    ("n_pooled_values", n_pooled_old),
]:
    if array.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected {name} shape: {array.shape}; "
            f"expected {expected_shape}."
        )
if not orientable:
    raise RuntimeError(
        "Expected orientable=True."
    )
if d_diff_max_separation != 2:
    raise RuntimeError(
        "Expected d_diff_max_separation=2."
    )
if d_min_joint_samples != 100:
    raise RuntimeError(
        f"Expected d_min_joint_samples=100, found "
        f"{d_min_joint_samples}."
    )
if nphi != 512:
    raise RuntimeError(
        f"Expected nphi=512, found {nphi}."
    )


# ==================================================
# DEFINE ORIGINAL MISSING CELLS
# ==================================================
requested_range = np.zeros(
    nphi + 1,
    dtype=bool,
)
requested_range[
    NJOINT_MIN:NJOINT_MAX + 1
] = True
original_target_mask = (
    (~fit_valid_old)
    & requested_range[None, :]
)
n_original_missing = int(
    np.count_nonzero(original_target_mask)
)


# ==================================================
# LOAD SUPPLEMENTAL GEOMETRY CATALOGUE
# ==================================================
geom = np.load(source_geometry_npz)
required_geometry_fields = [
    "alpha_index",
    "alpha_deg",
    "N_joint",
    "pixel_i",
    "pixel_j",
    "beta_bin",
    "beta_deg",
    "separation_deg",
    "original_target_mask",
    "found_count",
]

for field in required_geometry_fields:
    if field not in geom.files:
        raise RuntimeError(
            f"Geometry NPZ is missing field: {field}"
        )
    
geometry_alpha_index = np.asarray(
    geom["alpha_index"],
    dtype=np.int32,
)
geometry_alpha_deg = np.asarray(
    geom["alpha_deg"],
    dtype=np.float64,
)
geometry_n_joint = np.asarray(
    geom["N_joint"],
    dtype=np.int32,
)
geometry_pixel_i = np.asarray(
    geom["pixel_i"],
    dtype=np.int32,
)
geometry_pixel_j = np.asarray(
    geom["pixel_j"],
    dtype=np.int32,
)
geometry_beta_bin = np.asarray(
    geom["beta_bin"],
    dtype=np.int32,
)
geometry_beta_deg = np.asarray(
    geom["beta_deg"],
    dtype=np.float64,
)
geometry_separation_deg = np.asarray(
    geom["separation_deg"],
    dtype=np.float64,
)
geometry_original_target_mask = np.asarray(
    geom["original_target_mask"],
    dtype=bool,
)
found_count = np.asarray(
    geom["found_count"],
    dtype=np.int64,
)

n_geometry_records = len(geometry_alpha_index)
for name, array in [
    ("alpha_deg", geometry_alpha_deg),
    ("N_joint", geometry_n_joint),
    ("pixel_i", geometry_pixel_i),
    ("pixel_j", geometry_pixel_j),
    ("beta_bin", geometry_beta_bin),
    ("beta_deg", geometry_beta_deg),
    ("separation_deg", geometry_separation_deg),
]:
    if len(array) != n_geometry_records:
        raise RuntimeError(
            f"Geometry field {name} has inconsistent length."
        )
    
if geometry_original_target_mask.shape != expected_shape:
    raise RuntimeError(
        "Geometry original_target_mask has wrong shape."
    )
if found_count.shape != expected_shape:
    raise RuntimeError(
        "Geometry found_count has wrong shape."
    )
if not np.array_equal(
    geometry_original_target_mask,
    original_target_mask,
):
    raise RuntimeError(
        "Geometry catalogue target mask does not match the "
        "missing cells in the source Gaussian calibration."
    )

# Recount catalogue entries and require exact agreement with found_count.
recount = np.zeros(
    expected_shape,
    dtype=np.int64,
)
np.add.at(
    recount,
    (
        geometry_alpha_index,
        geometry_n_joint,
    ),
    1,
)
if not np.array_equal(
    recount,
    found_count,
):
    bad = np.argwhere(
        recount != found_count
    )[0]
    raise RuntimeError(
        "Geometry catalogue recount does not match found_count. "
        f"First mismatch: alpha_index={bad[0]}, "
        f"N_joint={bad[1]}, recount={recount[tuple(bad)]}, "
        f"found_count={found_count[tuple(bad)]}."
    )

# Every saved geometry must belong to an originally missing cell.
if n_geometry_records > 0:
    if not np.all(
        original_target_mask[
            geometry_alpha_index,
            geometry_n_joint,
        ]
    ):
        raise RuntimeError(
            "Geometry catalogue contains a record outside the "
            "originally missing cells."
        )
    
# Saved alpha / beta metadata must agree exactly with the indices.
if not np.allclose(
    geometry_alpha_deg,
    alpha_grid[geometry_alpha_index],
    rtol=0.0,
    atol=1.0e-10,
):
    raise RuntimeError(
        "Saved geometry alpha_deg does not match alpha_grid."
    )
expected_beta_deg = (
    360.0
    * geometry_beta_bin.astype(np.float64)
    / nphi
)
if not np.allclose(
    geometry_beta_deg,
    expected_beta_deg,
    rtol=0.0,
    atol=1.0e-10,
):
    raise RuntimeError(
        "Saved geometry beta_deg does not match beta_bin."
    )
if np.any(
    (geometry_beta_bin < 0)
    | (geometry_beta_bin >= nphi)
):
    raise RuntimeError(
        "Geometry catalogue contains an invalid beta_bin."
    )


# ==================================================
# IDENTIFY DIRECT-SUPPLEMENTAL VS FALLBACK CELLS
# ==================================================
supplemental_direct_mask = (
    original_target_mask
    & (found_count > 0)
)
zero_geometry_mask = (
    original_target_mask
    & (found_count == 0)
)
# Zero-geometry cells are handled by the generic fallback below.
zero_locations = np.argwhere(
    zero_geometry_mask
)
n_supplemental_cells = int(
    np.count_nonzero(
        supplemental_direct_mask
    )
)
n_zero_cells = int(
    np.count_nonzero(
        zero_geometry_mask
    )
)

print("======================================")
print("SUPPLEMENTAL GAUSSIAN COMPLETION")
print("======================================")
print(f"alpha values                 : {n_alpha}")
print(f"N_joint range                : {NJOINT_MIN} to {NJOINT_MAX}")
print(f"original missing cells       : {n_original_missing}")
print(f"direct supplemental cells    : {n_supplemental_cells}")
print(f"fallback cells               : {n_zero_cells}")
print(f"saved supplemental geometries: {n_geometry_records}")
print(f"null maps                    : {len(null_files)}")
print()


# ==================================================
# READ MASK AND D FILTER
# ==================================================
obs_mask = hp.read_map(
    mask_file,
    field=0,
    dtype=np.float64,
)
if hp.get_nside(obs_mask) != nside:
    raise RuntimeError(
        f"Mask NSIDE={hp.get_nside(obs_mask)}, expected {nside}."
    )
filter_table = np.loadtxt(
    filter_file,
    comments="#",
)
d_filter = np.zeros(
    d_filter_lmax + 1,
    dtype=np.float64,
)
filter_ell = filter_table[:, 0].astype(int)
filter_value = np.asarray(
    filter_table[:, 1],
    dtype=np.float64,
)
inside = (
    (filter_ell >= 0)
    & (filter_ell <= d_filter_lmax)
)
d_filter[
    filter_ell[inside]
] = filter_value[inside]
def apply_d_filter(input_map):
    """Exactly the same map filter used in the original calibration."""
    alm = hp.map2alm(
        input_map,
        lmax=d_filter_lmax,
    )
    filtered_alm = hp.almxfl(
        alm,
        d_filter,
    )
    return hp.alm2map(
        filtered_alm,
        nside=nside,
        lmax=d_filter_lmax,
    )


# ==================================================
# CIRCLE GEOMETRY
# ==================================================
phi_ring = (
    2.0
    * np.pi
    * np.arange(
        1,
        nphi + 1,
        dtype=np.float64,
    )
    / nphi
)
def circle_ring_pixels(
    pixel,
    alpha_deg,
):
    theta, phi = hp.pix2ang(
        nside_search,
        pixel,
        nest=False,
    )
    alpha = np.deg2rad(
        alpha_deg
    )
    rotation_matrix = np.array(
        [
            [
                np.cos(theta) * np.cos(phi),
                -np.sin(phi),
                np.sin(theta) * np.cos(phi),
            ],
            [
                np.cos(theta) * np.sin(phi),
                np.cos(phi),
                np.sin(theta) * np.sin(phi),
            ],
            [
                -np.sin(theta),
                0.0,
                np.cos(theta),
            ],
        ],
        dtype=np.float64,
    )
    local_vectors = np.vstack(
        [
            np.sin(alpha) * np.cos(phi_ring),
            np.sin(alpha) * np.sin(phi_ring),
            np.full(
                nphi,
                np.cos(alpha),
                dtype=np.float64,
            ),
        ]
    )
    sky_vectors = (
        rotation_matrix
        @ local_vectors
    )
    theta_ring = np.arccos(
        np.clip(
            sky_vectors[2],
            -1.0,
            1.0,
        )
    )
    phi_ring_sky = (
        np.arctan2(
            sky_vectors[1],
            sky_vectors[0],
        )
        + np.pi
    )
    return hp.ang2pix(
        nside,
        theta_ring,
        phi_ring_sky,
        nest=False,
    )


# ==================================================
# PRECOMPUTE THE RING PIXELS / MASK VALIDITY USED BY
# THE SUPPLEMENTAL GEOMETRY CATALOGUE
# ==================================================
circle_keys = set()
for alpha_index, pixel_i, pixel_j in zip(
    geometry_alpha_index,
    geometry_pixel_i,
    geometry_pixel_j,
):
    circle_keys.add(
        (
            int(alpha_index),
            int(pixel_i),
        )
    )
    circle_keys.add(
        (
            int(alpha_index),
            int(pixel_j),
        )
    )
circle_cache = {}
for cache_index, (
    alpha_index,
    pixel,
) in enumerate(
    sorted(circle_keys)
):
    ring_pixels = circle_ring_pixels(
        pixel,
        float(alpha_grid[alpha_index]),
    )
    valid = (
        obs_mask[ring_pixels]
        >= mask_threshold
    )
    circle_cache[
        (alpha_index, pixel)
    ] = (
        ring_pixels,
        valid,
    )
    if (
        cache_index == 0
        or (cache_index + 1) % 1000 == 0
        or cache_index + 1 == len(circle_keys)
    ):
        print(
            f"Precomputed circles: {cache_index + 1}/"
            f"{len(circle_keys)}"
        )
print()


# ==================================================
# VERIFY EVERY SAVED (i,j,beta|alpha) REALLY HAS THE
# SAVED N_joint, USING A DIRECT MASK-ONLY CALCULATION.
# ==================================================
index_ring = np.arange(
    nphi,
    dtype=np.int64,
)
def n_joint_at_beta_from_validity(
    valid_i,
    valid_j,
    beta_bin,
):
    partner = (
        beta_bin
        - index_ring
    ) % nphi
    n_joint = nphi
    for separation_bin in range(
        1,
        d_diff_max_separation + 1,
    ):
        valid_delta_i = (
            valid_i
            & np.roll(
                valid_i,
                -separation_bin,
            )
        )
        valid_delta_j = (
            valid_j
            & np.roll(
                valid_j,
                separation_bin,
            )
        )
        n_joint_s = int(
            np.count_nonzero(
                valid_delta_i
                & valid_delta_j[partner]
            )
        )
        n_joint = min(
            n_joint,
            n_joint_s,
        )
    return n_joint
for record_index in range(
    n_geometry_records
):
    alpha_index = int(
        geometry_alpha_index[record_index]
    )
    pixel_i = int(
        geometry_pixel_i[record_index]
    )
    pixel_j = int(
        geometry_pixel_j[record_index]
    )
    beta_bin = int(
        geometry_beta_bin[record_index]
    )
    target_n = int(
        geometry_n_joint[record_index]
    )
    _, valid_i = circle_cache[
        (alpha_index, pixel_i)
    ]
    _, valid_j = circle_cache[
        (alpha_index, pixel_j)
    ]
    actual_n = n_joint_at_beta_from_validity(
        valid_i,
        valid_j,
        beta_bin,
    )
    if actual_n != target_n:
        raise RuntimeError(
            "Saved supplemental geometry failed direct N_joint "
            "verification: "
            f"record={record_index}, alpha_index={alpha_index}, "
            f"N_saved={target_n}, N_actual={actual_n}, "
            f"pixel_i={pixel_i}, pixel_j={pixel_j}, "
            f"beta_bin={beta_bin}."
        )
print(
    "Supplemental geometry N_joint verification: PASS "
    f"({n_geometry_records} records)"
)
print()


# ==================================================
# GROUP SAVED RECORDS BY (alpha_index, pixel_i, pixel_j)
# SO ONE CIRCLE-PAIR EVALUATION CAN SERVE EVERY SAVED
# BETA FOR THAT PAIR / ALPHA.
# ==================================================
group_records = defaultdict(list)
for record_index in range(
    n_geometry_records
):
    key = (
        int(geometry_alpha_index[record_index]),
        int(geometry_pixel_i[record_index]),
        int(geometry_pixel_j[record_index]),
    )
    group_records[key].append(
        record_index
    )
# Deterministic processing order.
groups = []
for key in sorted(group_records):
    record_indices = np.asarray(
        group_records[key],
        dtype=np.int64,
    )
    # Within a given pair+alpha, each target cell should occur once.
    n_values = geometry_n_joint[
        record_indices
    ]
    if len(np.unique(n_values)) != len(n_values):
        raise RuntimeError(
            "Duplicate N_joint target within one "
            "(alpha,pixel_i,pixel_j) group."
        )
    groups.append(
        (
            key,
            record_indices,
        )
    )
print(
    f"Unique supplemental (alpha,i,j) groups: {len(groups)}"
)
print()


# ==================================================
# TEMPERATURE RING PROCESSING
# ==================================================
def processed_ring_from_cache(
    sky_map,
    alpha_index,
    pixel,
):
    ring_pixels, valid = circle_cache[
        (alpha_index, pixel)
    ]
    temperature = np.asarray(
        sky_map[ring_pixels],
        dtype=np.float64,
    )
    # Same low-mode removal as the original calibration.
    modes = np.fft.rfft(
        temperature
    )
    modes[:m_min] = 0.0
    temperature = np.fft.irfft(
        modes,
        n=nphi,
    )
    return temperature, valid


# ==================================================
# DIRECT D_diff AT ONLY THE SAVED BETA BINS
#
# This is algebraically identical to selecting those beta bins
# from the full FFT D_diff curve, but avoids calculating all 512
# beta values when only a few are needed.
# ==================================================
def get_d_diff_at_selected_betas(
    ring_i,
    valid_i,
    ring_j,
    valid_j,
    beta_bins,
):
    beta_bins = np.asarray(
        beta_bins,
        dtype=np.int64,
    )
    n_beta = len(beta_bins)
    numerator_total = np.zeros(
        n_beta,
        dtype=np.float64,
    )
    denominator_total = np.zeros(
        n_beta,
        dtype=np.float64,
    )
    n_joint = np.full(
        n_beta,
        nphi,
        dtype=np.int16,
    )
    partners = (
        beta_bins[:, None]
        - index_ring[None, :]
    ) % nphi
    for separation_bin in range(
        1,
        d_diff_max_separation + 1,
    ):
        delta_t_i = (
            np.roll(
                ring_i,
                -separation_bin,
            )
            - ring_i
        )
        valid_delta_i = (
            valid_i
            & np.roll(
                valid_i,
                -separation_bin,
            )
        )
        # Current calibration is orientable=True.
        delta_t_j = (
            np.roll(
                ring_j,
                separation_bin,
            )
            - ring_j
        )
        valid_delta_j = (
            valid_j
            & np.roll(
                valid_j,
                separation_bin,
            )
        )
        for beta_local in range(
            n_beta
        ):
            partner = partners[
                beta_local
            ]
            joint = (
                valid_delta_i
                & valid_delta_j[partner]
            )
            n_joint_s = int(
                np.count_nonzero(joint)
            )
            n_joint[beta_local] = min(
                int(n_joint[beta_local]),
                n_joint_s,
            )
            if n_joint_s == 0:
                continue
            partner_joint = partner[
                joint
            ]
            delta_i_joint = delta_t_i[
                joint
            ]
            delta_j_joint = delta_t_j[
                partner_joint
            ]
            numerator_total[
                beta_local
            ] += (
                2.0
                * np.sum(
                    delta_i_joint
                    * delta_j_joint
                )
            )
            denominator_total[
                beta_local
            ] += np.sum(
                delta_i_joint**2
                + delta_j_joint**2
            )
    d_diff = np.full(
        n_beta,
        np.nan,
        dtype=np.float64,
    )
    usable = (
        (n_joint >= d_min_joint_samples)
        & (
            denominator_total
            > np.finfo(np.float64).tiny
        )
    )
    d_diff[usable] = np.clip(
        numerator_total[usable]
        / denominator_total[usable],
        -1.0,
        1.0,
    )
    return d_diff, n_joint


# ==================================================
# FULL FFT REFERENCE CURVE, USED ONLY FOR A STARTUP
# SELF-CHECK OF THE DIRECT SELECTED-BETA EVALUATOR.
# ==================================================
def get_d_diff_curve_fft_from_rings(
    ring_i,
    valid_i,
    ring_j,
    valid_j,
):
    numerator_total = np.zeros(
        nphi,
        dtype=np.float64,
    )
    denominator_total = np.zeros(
        nphi,
        dtype=np.float64,
    )
    n_joint = np.full(
        nphi,
        nphi,
        dtype=np.int16,
    )
    for separation_bin in range(
        1,
        d_diff_max_separation + 1,
    ):
        delta_t_i = (
            np.roll(
                ring_i,
                -separation_bin,
            )
            - ring_i
        )
        valid_delta_i = (
            valid_i
            & np.roll(
                valid_i,
                -separation_bin,
            )
        )
        delta_t_j = (
            np.roll(
                ring_j,
                separation_bin,
            )
            - ring_j
        )
        valid_delta_j = (
            valid_j
            & np.roll(
                valid_j,
                separation_bin,
            )
        )
        w_i = valid_delta_i.astype(
            np.float64
        )
        w_j = valid_delta_j.astype(
            np.float64
        )
        fw_i = np.fft.fft(w_i)
        fw_j = np.fft.fft(w_j)
        fx_i = np.fft.fft(
            w_i * delta_t_i
        )
        fx_j = np.fft.fft(
            w_j * delta_t_j
        )
        fe_i = np.fft.fft(
            w_i * delta_t_i**2
        )
        fe_j = np.fft.fft(
            w_j * delta_t_j**2
        )
        numerator_total += (
            2.0
            * np.fft.ifft(
                fx_i * fx_j
            ).real
        )
        denominator_total += np.fft.ifft(
            fe_i * fw_j
            + fw_i * fe_j
        ).real
        n_joint_s = np.rint(
            np.fft.ifft(
                fw_i * fw_j
            ).real
        ).astype(np.int16)
        n_joint_s = np.clip(
            n_joint_s,
            0,
            nphi,
        )
        n_joint = np.minimum(
            n_joint,
            n_joint_s,
        )
    d_diff = np.full(
        nphi,
        np.nan,
        dtype=np.float64,
    )
    usable = (
        (n_joint >= d_min_joint_samples)
        & (
            denominator_total
            > np.finfo(np.float64).tiny
        )
    )
    d_diff[usable] = np.clip(
        numerator_total[usable]
        / denominator_total[usable],
        -1.0,
        1.0,
    )
    return d_diff, n_joint


# ==================================================
# GEOMETRY SIGNATURE FOR SAFE CHECKPOINT RESUME
# ==================================================
signature_hasher = hashlib.sha256()
for array in [
    geometry_alpha_index,
    geometry_n_joint,
    geometry_pixel_i,
    geometry_pixel_j,
    geometry_beta_bin,
]:
    signature_hasher.update(
        np.ascontiguousarray(array).tobytes()
    )
geometry_signature = (
    signature_hasher.hexdigest()
)


# ==================================================
# SUPPLEMENTAL ONLINE MOMENTS
# ==================================================
supp_count = np.zeros(
    expected_shape,
    dtype=np.int64,
)
supp_sum = np.zeros(
    expected_shape,
    dtype=np.float64,
)
supp_sumsq = np.zeros(
    expected_shape,
    dtype=np.float64,
)
start_null_index = 0

if checkpoint_npz.exists():
    checkpoint = np.load(
        checkpoint_npz
    )
    checkpoint_signature = str(
        checkpoint["geometry_signature"]
    )
    if checkpoint_signature != geometry_signature:
        raise RuntimeError(
            "Checkpoint geometry signature does not match the "
            "current supplemental geometry catalogue."
        )
    
    supp_count = np.asarray(
        checkpoint["count"],
        dtype=np.int64,
    )
    supp_sum = np.asarray(
        checkpoint["sum"],
        dtype=np.float64,
    )
    supp_sumsq = np.asarray(
        checkpoint["sumsq"],
        dtype=np.float64,
    )
    start_null_index = int(
        checkpoint["next_null_index"]
    )

    if supp_count.shape != expected_shape:
        raise RuntimeError(
            "Checkpoint count shape is wrong."
        )
    if supp_sum.shape != expected_shape:
        raise RuntimeError(
            "Checkpoint sum shape is wrong."
        )
    if supp_sumsq.shape != expected_shape:
        raise RuntimeError(
            "Checkpoint sumsq shape is wrong."
        )
    if not (
        0 <= start_null_index <= len(null_files)
    ):
        raise RuntimeError(
            "Checkpoint next_null_index is invalid."
        )
    print(
        f"Resuming checkpoint at null map "
        f"{start_null_index + 1}/{len(null_files)}"
        if start_null_index < len(null_files)
        else "Checkpoint already contains all null maps."
    )
    print()


# ==================================================
# FIRST-NULL SELF-CHECK:
# DIRECT SELECTED-BETA D_diff == FULL FFT D_diff
# ==================================================
if start_null_index == 0:
    first_null_map = hp.read_map(
        null_files[0],
        field=0,
        dtype=np.float64,
    )
    first_filtered_null = apply_d_filter(
        first_null_map
    )
    n_check_groups = min(
        10,
        len(groups),
    )
    for check_group_index in range(
        n_check_groups
    ):
        (
            alpha_index,
            pixel_i,
            pixel_j,
        ), record_indices = groups[
            check_group_index
        ]
        ring_i, valid_i = processed_ring_from_cache(
            first_filtered_null,
            alpha_index,
            pixel_i,
        )
        ring_j, valid_j = processed_ring_from_cache(
            first_filtered_null,
            alpha_index,
            pixel_j,
        )
        beta_bins = geometry_beta_bin[
            record_indices
        ]
        direct_d, direct_n = get_d_diff_at_selected_betas(
            ring_i,
            valid_i,
            ring_j,
            valid_j,
            beta_bins,
        )
        fft_d, fft_n = get_d_diff_curve_fft_from_rings(
            ring_i,
            valid_i,
            ring_j,
            valid_j,
        )
        target_n = geometry_n_joint[
            record_indices
        ]
        if not np.array_equal(
            direct_n,
            target_n,
        ):
            raise RuntimeError(
                "Direct selected-beta N_joint self-check failed."
            )
        if not np.array_equal(
            fft_n[beta_bins],
            target_n,
        ):
            raise RuntimeError(
                "FFT-reference N_joint self-check failed."
            )
        if not np.allclose(
            direct_d,
            fft_d[beta_bins],
            rtol=1.0e-12,
            atol=1.0e-12,
            equal_nan=True,
        ):
            raise RuntimeError(
                "Direct selected-beta D_diff does not match the "
                "full FFT reference curve."
            )
    print(
        f"Direct-vs-FFT D_diff self-check: PASS "
        f"({n_check_groups} groups)"
    )
    print()
else:
    first_filtered_null = None


# ==================================================
# 500 NULL MAPS: FIT ONLY THE SUPPLEMENTAL DIRECT CELLS
# ==================================================
for null_index in range(
    start_null_index,
    len(null_files),
):
    if (
        null_index == 0
        and first_filtered_null is not None
    ):
        filtered_null_map = (
            first_filtered_null
        )
    else:
        null_map = hp.read_map(
            null_files[null_index],
            field=0,
            dtype=np.float64,
        )
        filtered_null_map = apply_d_filter(
            null_map
        )
    for (
        alpha_index,
        pixel_i,
        pixel_j,
    ), record_indices in groups:
        ring_i, valid_i = processed_ring_from_cache(
            filtered_null_map,
            alpha_index,
            pixel_i,
        )
        ring_j, valid_j = processed_ring_from_cache(
            filtered_null_map,
            alpha_index,
            pixel_j,
        )
        beta_bins = geometry_beta_bin[
            record_indices
        ]
        target_n = geometry_n_joint[
            record_indices
        ]
        d_values, actual_n = get_d_diff_at_selected_betas(
            ring_i,
            valid_i,
            ring_j,
            valid_j,
            beta_bins,
        )

        # Mask geometry is map-independent, so this must always agree.
        if not np.array_equal(
            actual_n,
            target_n,
        ):
            raise RuntimeError(
                "N_joint changed unexpectedly during null processing."
            )
        finite = np.isfinite(
            d_values
        )
        if not np.any(finite):
            continue
        n_use = target_n[
            finite
        ].astype(
            np.int64,
            copy=False,
        )
        d_use = d_values[
            finite
        ]

        # A group contains at most one record for each N_joint,
        # so direct indexed addition is safe here.
        supp_count[
            alpha_index,
            n_use,
        ] += 1
        supp_sum[
            alpha_index,
            n_use,
        ] += d_use
        supp_sumsq[
            alpha_index,
            n_use,
        ] += d_use**2
    completed_nulls = (
        null_index + 1
    )
    if (
        completed_nulls == 1
        or completed_nulls % CHECKPOINT_EVERY == 0
        or completed_nulls == len(null_files)
    ):
        np.savez_compressed(
            checkpoint_npz,
            geometry_signature=np.array(
                geometry_signature
            ),
            next_null_index=np.array(
                completed_nulls
            ),
            count=supp_count,
            sum=supp_sum,
            sumsq=supp_sumsq,
            source_geometry_npz=np.array(
                str(source_geometry_npz)
            ),
            source_calibration_npz=np.array(
                str(source_calibration_npz)
            ),
        )
        print(
            f"Null maps: {completed_nulls}/{len(null_files)} "
            f"(checkpoint saved)"
        )


# ==================================================
# FIT SUPPLEMENTAL DIRECT GAUSSIANS
# ==================================================
supp_mu = np.full(
    expected_shape,
    np.nan,
    dtype=np.float64,
)
supp_sigma = np.full(
    expected_shape,
    np.nan,
    dtype=np.float64,
)
supp_valid = (
    supplemental_direct_mask
    & (supp_count >= 3)
)
supp_mu[supp_valid] = (
    supp_sum[supp_valid]
    / supp_count[supp_valid]
)
supp_variance = np.full(
    expected_shape,
    np.nan,
    dtype=np.float64,
)
supp_variance[supp_valid] = (
    supp_sumsq[supp_valid]
    / supp_count[supp_valid]
    - supp_mu[supp_valid]**2
)

# Remove only tiny negative floating-point roundoff.
supp_variance[supp_valid] = np.maximum(
    supp_variance[supp_valid],
    0.0,
)
supp_sigma[supp_valid] = np.sqrt(
    supp_variance[supp_valid]
)
supp_valid &= (
    np.isfinite(supp_mu)
    & np.isfinite(supp_sigma)
    & (supp_sigma > 0.0)
)
missing_direct_fit = (
    supplemental_direct_mask
    & (~supp_valid)
)
if np.any(missing_direct_fit):
    bad = np.argwhere(
        missing_direct_fit
    )[0]
    raise RuntimeError(
        "A reachable supplemental cell failed Gaussian fitting: "
        f"alpha_index={bad[0]}, N_joint={bad[1]}, "
        f"count={supp_count[tuple(bad)]}."
    )


# ==================================================
# BUILD COMPLETE TABLE
#
# fit_method:
#   0 = original direct fit
#   1 = supplemental direct fit
#   2 = interpolated/extrapolated fallback for zero-geometry cells
# ==================================================
mu_final = mu_old.copy()
sigma_final = sigma_old.copy()
fit_valid_final = fit_valid_old.copy()
n_geometry_final = n_geometry_old.copy()
n_pooled_final = n_pooled_old.copy()
fit_method = np.full(
    expected_shape,
    -1,
    dtype=np.int8,
)
fit_method[
    fit_valid_old
    & requested_range[None, :]
] = 0
# Insert direct supplemental fits.
mu_final[
    supplemental_direct_mask
] = supp_mu[
    supplemental_direct_mask
]
sigma_final[
    supplemental_direct_mask
] = supp_sigma[
    supplemental_direct_mask
]
fit_valid_final[
    supplemental_direct_mask
] = True
n_geometry_final[
    supplemental_direct_mask
] = found_count[
    supplemental_direct_mask
]
n_pooled_final[
    supplemental_direct_mask
] = supp_count[
    supplemental_direct_mask
]
fit_method[
    supplemental_direct_mask
] = 1


# ==================================================
# FALLBACK FOR ZERO-GEOMETRY CELLS
#
# Use the nearest valid direct/supplemental N_joint cells at the same alpha.
# If valid cells exist on both sides, interpolate linearly in N_joint.
# At a boundary (for example N_joint=512), use the two nearest valid cells
# on the available side for linear extrapolation.
#
# The Gaussian mean and variance are interpolated/extrapolated separately.
# Fallback cells are explicitly marked fit_method=2 and have zero direct
# geometry / pooled-null counts.
# ==================================================
for alpha_index, n_joint_value in zero_locations:
    alpha_index = int(alpha_index)
    n_joint_value = int(n_joint_value)
    direct_valid = (
        fit_valid_final[alpha_index]
        & (fit_method[alpha_index] >= 0)
        & (fit_method[alpha_index] <= 1)
    )
    valid_n = np.flatnonzero(
        direct_valid
        & requested_range
    )
    below = valid_n[
        valid_n < n_joint_value
    ]
    above = valid_n[
        valid_n > n_joint_value
    ]
    if len(below) > 0 and len(above) > 0:
        n0 = int(below[-1])
        n1 = int(above[0])
    elif len(below) >= 2:
        n0 = int(below[-2])
        n1 = int(below[-1])
    elif len(above) >= 2:
        n0 = int(above[0])
        n1 = int(above[1])
    else:
        raise RuntimeError(
            "Cannot construct fallback because fewer than two "
            "direct valid N_joint cells are available at "
            f"alpha_index={alpha_index}, N_joint={n_joint_value}."
        )
    weight = (
        (n_joint_value - n0)
        / float(n1 - n0)
    )
    mu0 = mu_final[
        alpha_index,
        n0,
    ]
    mu1 = mu_final[
        alpha_index,
        n1,
    ]
    var0 = sigma_final[
        alpha_index,
        n0,
    ]**2
    var1 = sigma_final[
        alpha_index,
        n1,
    ]**2
    mu_fallback = (
        (1.0 - weight) * mu0
        + weight * mu1
    )
    variance_fallback = (
        (1.0 - weight) * var0
        + weight * var1
    )
    if not (
        np.isfinite(mu_fallback)
        and np.isfinite(variance_fallback)
        and variance_fallback > 0.0
    ):
        raise RuntimeError(
            "Invalid fallback Gaussian at "
            f"alpha_index={alpha_index}, "
            f"N_joint={n_joint_value}, "
            f"sources=({n0},{n1}), "
            f"variance={variance_fallback}."
        )
    mu_final[
        alpha_index,
        n_joint_value,
    ] = mu_fallback
    sigma_final[
        alpha_index,
        n_joint_value,
    ] = np.sqrt(
        variance_fallback
    )
    fit_valid_final[
        alpha_index,
        n_joint_value,
    ] = True
    n_geometry_final[
        alpha_index,
        n_joint_value,
    ] = 0
    n_pooled_final[
        alpha_index,
        n_joint_value,
    ] = 0
    fit_method[
        alpha_index,
        n_joint_value,
    ] = 2
# ==================================================
# FINAL COMPLETENESS CHECK
# ==================================================
final_requested_valid = fit_valid_final[
    :,
    NJOINT_MIN:NJOINT_MAX + 1,
]
if not np.all(
    final_requested_valid
):
    bad_local = np.argwhere(
        ~final_requested_valid
    )[0]
    bad_alpha = int(
        bad_local[0]
    )
    bad_n = int(
        NJOINT_MIN
        + bad_local[1]
    )
    raise RuntimeError(
        "Final table is not complete: "
        f"alpha_index={bad_alpha}, N_joint={bad_n}."
    )
for array_name, array in [
    ("mu", mu_final),
    ("sigma", sigma_final),
]:
    requested_values = array[
        :,
        NJOINT_MIN:NJOINT_MAX + 1,
    ]
    if not np.all(
        np.isfinite(requested_values)
    ):
        raise RuntimeError(
            f"Final {array_name} contains non-finite requested values."
        )
if not np.all(
    sigma_final[
        :,
        NJOINT_MIN:NJOINT_MAX + 1,
    ] > 0.0
):
    raise RuntimeError(
        "Final sigma contains a non-positive requested value."
    )
if np.any(
    fit_method[
        :,
        NJOINT_MIN:NJOINT_MAX + 1,
    ] < 0
):
    raise RuntimeError(
        "Final table has an unassigned fit_method."
    )


# ==================================================
# SAVE COMPLETE MACHINE-READABLE TABLE
# ==================================================
# Preserve the original metadata, then replace the lookup arrays
# with the completed versions and add explicit provenance fields.
payload = {
    key: old[key]
    for key in old.files
}
payload.update(
    {
        "mu": mu_final,
        "sigma": sigma_final,
        "fit_valid": fit_valid_final,
        "n_geometry": n_geometry_final,
        "n_pooled_values": n_pooled_final,
        "fit_method": fit_method,
        "source_fit_valid": fit_valid_old,
        "source_mu": mu_old,
        "source_sigma": sigma_old,
        "supplemental_count": supp_count,
        "supplemental_sum": supp_sum,
        "supplemental_sumsq": supp_sumsq,
        "supplemental_geometry_count": found_count,
        "source_calibration_npz": np.array(
            str(source_calibration_npz)
        ),
        "source_geometry_npz": np.array(
            str(source_geometry_npz)
        ),
        "completion_method": np.array(
            "original direct fits + supplemental direct fits + "
            "nearest-valid linear fallback for zero-geometry cells"
        ),
        "fit_method_0": np.array(
            "original_direct"
        ),
        "fit_method_1": np.array(
            "supplemental_direct"
        ),
        "fit_method_2": np.array(
            "nearest_valid_linear_fallback"
        ),
        "fallback_interpolates": np.array(
            "mu and variance"
        ),
        "completed_n_null_maps": np.array(
            len(null_files)
        ),
        "completed_table": np.array(True),
    }
)
np.savez_compressed(
    output_npz,
    **payload,
)


# ==================================================
# SAVE HUMAN-READABLE COMPLETE TABLE
# ==================================================
rows = []
for alpha_index, alpha_deg in enumerate(
    alpha_grid
):
    for n_joint_value in range(
        NJOINT_MIN,
        NJOINT_MAX + 1,
    ):
        rows.append(
            [
                alpha_index,
                alpha_deg,
                n_joint_value,
                mu_final[
                    alpha_index,
                    n_joint_value,
                ],
                sigma_final[
                    alpha_index,
                    n_joint_value,
                ],
                n_geometry_final[
                    alpha_index,
                    n_joint_value,
                ],
                n_pooled_final[
                    alpha_index,
                    n_joint_value,
                ],
                int(
                    fit_valid_final[
                        alpha_index,
                        n_joint_value,
                    ]
                ),
                int(
                    fit_method[
                        alpha_index,
                        n_joint_value,
                    ]
                ),
            ]
        )
rows = np.asarray(
    rows,
    dtype=np.float64,
)
np.savetxt(
    output_table,
    rows,
    fmt=[
        "%d",
        "%.12f",
        "%d",
        "%.16e",
        "%.16e",
        "%d",
        "%d",
        "%d",
        "%d",
    ],
    header=(
        "alpha_index alpha_deg N_joint mu sigma "
        "n_geometry n_pooled_values fit_valid fit_method\n"
        "fit_method: 0=original_direct, 1=supplemental_direct, "
        "2=nearest_valid_linear_fallback\n"
        "For fit_method=2, n_geometry=n_pooled_values=0 by design."
    ),
)
# ==================================================
# CONCISE FINAL SUMMARY
# ==================================================
method_counts = {
    method: int(
        np.count_nonzero(
            fit_method[
                :,
                NJOINT_MIN:NJOINT_MAX + 1,
            ]
            == method
        )
    )
    for method in (0, 1, 2)
}
supp_geometry_counts = found_count[
    supplemental_direct_mask
]
supp_pooled_counts = supp_count[
    supplemental_direct_mask
]

print()
print("======================================")
print("COMPLETE GAUSSIAN LOOKUP SAVED")
print("======================================")
print(
    f"requested cells       : "
    f"{n_alpha * (NJOINT_MAX - NJOINT_MIN + 1)}"
)
print(
    f"original direct       : {method_counts[0]}"
)
print(
    f"supplemental direct   : {method_counts[1]}"
)
print(
    f"fallback zero-geometry: {method_counts[2]}"
)
print(
    "supplemental geometries/cell min/median/max: "
    f"{int(np.min(supp_geometry_counts))}/"
    f"{float(np.median(supp_geometry_counts)):.1f}/"
    f"{int(np.max(supp_geometry_counts))}"
)
print(
    "supplemental null values/cell min/median/max: "
    f"{int(np.min(supp_pooled_counts))}/"
    f"{float(np.median(supp_pooled_counts)):.1f}/"
    f"{int(np.max(supp_pooled_counts))}"
)

print("all N_joint=100..512 complete: YES")
print()
print("Saved NPZ  :", output_npz)
print("Saved table:", output_table)
print("Checkpoint :", checkpoint_npz)